In [1]:
import os, pydicom, random, math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from xgboost import XGBRegressor

DATA_PATH = "../input/osic-pulmonary-fibrosis-progression/"
VALID_SPLIT = 0.2
MIN_TEST_WEEK = -12
MAX_TEST_WEEK = 133
TARGET_VAR = 'deltaFVC'

# Image processing
MAX_PLANES, MAX_ROW, MAX_COL = 10, 100, 100   # Maximum dimensions on Z-, X- and Y- axes
INVERSE_WEIGHT_FCT = lambda y: y              # Inverse distance penalty function for weighting
N_NEIGHBOURS = 2
PIXEL_VALUE_RANGE = 32747 + 15000

def combine_duplicates(df, FUN=np.mean):
    df = df.assign(patientWeeks = df.Patient + "__" + df.Weeks.astype(str))
    table = df.patientWeeks.value_counts()
    duplicates = table.loc[table > 1].index
    subset = df.loc[df.patientWeeks.isin(duplicates)]
    avgFVC = subset.groupby(['Patient', 'Weeks']).FVC.agg(FUN)
    avgPct = subset.groupby(['Patient', 'Weeks']).Percent.agg(FUN)    
    subset = subset.drop_duplicates(subset=['Patient', 'Weeks']
                                   ).drop(labels=['FVC', 'Percent'], axis=1)
    subset = subset.join(avgFVC, on=['Patient', 'Weeks']
                        ).join(avgPct, on=['Patient', 'Weeks'])
    df = pd.concat([df[~df.patientWeeks.isin(subset.patientWeeks)],
                    subset]).sort_values(by=['Patient', 'Weeks'])
    return df.drop(labels=['patientWeeks'], axis=1)

# Interpolate the progression of FVC and Percent between weeks with data
def interpolate(df):
    ids = np.unique(df.Patient)
    df_mod = pd.DataFrame()
    for i in range(len(ids)):
        subset = df.loc[df.Patient == ids[i]]
        df_mod = pd.concat([df_mod, subset])
        age, sex, smSt = subset.iloc[0].loc[['Age', 'Sex', 'SmokingStatus']]
        for t in range(subset.shape[0]-1):
            gap = subset.Weeks.iloc[t+1] - subset.Weeks.iloc[t]
            if gap > 1:
                base_Week, base_FVC, base_Pct = subset.iloc[t].loc[['Weeks', 'FVC', 
                                                                    'Percent']]
                end_FVC, end_Pct = subset.iloc[t+1].loc[['FVC', 'Percent']]
                for j in range(1, gap):
                    new = pd.DataFrame({'Patient': ids[i],
                                        'Weeks': base_Week + j,
                                        'FVC': base_FVC + j / gap * (end_FVC - base_FVC),
                                        'Percent' : base_Pct + j / gap * (end_Pct - base_Pct),
                                        'Age': age, 'Sex': sex, 'SmokingStatus': smSt
                                       }, index=[None])
                    df_mod = pd.concat([df_mod, new])
    return df_mod

def compute_deltas(df):
    df = df.sort_values(by=['Weeks', 'Patient']).reset_index().drop('index', axis=1)
    deltas, idx = [], []
    for i, row in df.iterrows():
        patient, week = row.Patient, row.Weeks
        nextweek = df.loc[(df.Patient == patient) & (df.Weeks == week+1)]
        if len(nextweek) == 1:
            deltas = np.concatenate([deltas, (nextweek.FVC / row.FVC - 1)])
            idx.append(i)
    df = df.join(pd.DataFrame({'deltaFVC': deltas}, index=idx))
    return df.drop(np.where(np.isnan(df.deltaFVC))[0], axis=0)

class CSVDataPrep():
    
    def __init__(self, data_path, valid_split):
        data = pd.read_csv(data_path)
        data = combine_duplicates(data)
        data = interpolate(data)
        data = compute_deltas(data)
        self.split_valid(data, valid_split)        
        
    def split_valid(self, data, valid_split):
        patients = np.array(data.Patient.unique())
        valid = random.sample(list(patients), int(len(patients) * valid_split))
        train = patients[~np.in1d(patients, valid)]
        self.train = data.loc[data.Patient.isin(train)].reset_index(drop=True)
        self.valid = data.loc[data.Patient.isin(valid)].reset_index(drop=True)
        
    def pull(self):
        return self.train, self.valid
        
train, valid = CSVDataPrep(DATA_PATH + "train.csv", VALID_SPLIT).pull()
print(train.shape)
print(valid.shape)
train.head()


/tmp/ipykernel_11/4127862789.py:29: FutureWarning: The provided callable <function mean at 0x7ffff009d620> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  avgFVC = subset.groupby(['Patient', 'Weeks']).FVC.agg(FUN)
/tmp/ipykernel_11/4127862789.py:30: FutureWarning: The provided callable <function mean at 0x7ffff009d620> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  avgPct = subset.groupby(['Patient', 'Weeks']).Percent.agg(FUN)


(6582, 8)
(1690, 8)


,Patient,Weeks,FVC,Percent,Age,Sex,SmokingStatus,deltaFVC
0,ID00086637202203494931510,-5,3367.000000,117.628563,65,Female,Never smoked,-0.001728
1,ID00076637202199015035026,-4,2298.000000,52.749977,51,Male,Never smoked,0.017282
2,ID00086637202203494931510,-4,3361.181818,117.425301,65,Female,Never smoked,-0.001731
3,ID00023637202179104603099,-3,1536.000000,65.306122,71,Female,Ex-smoker,-0.018229
4,ID00076637202199015035026,-3,2337.714286,53.661608,51,Male,Never smoked,0.016989


In [2]:
class FibrosisModel():
    
    
    def __init__(self, model_type, img_data=None, kernels=10, 
                 y=TARGET_VAR, **kwargs):
        self.y = y
        self.model = model_type(**kwargs)
        self.img_data = img_data
        self.cat_encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)
        self.scaler = StandardScaler()
        self.firstrun = True
        
        
    def sampler(self, data, sample_size):
        output = pd.DataFrame()
        big_urn = np.array(data.index)
        maxes = data.groupby('Patient').Weeks.agg(max)
        adjust = len(big_urn) / (len(big_urn) - len(maxes))
        for i in range(int(sample_size * adjust)):
            choice = random.choice(big_urn)
            draw = data.iloc[choice]
            patient = draw.Patient
            curWeek = draw.Weeks
            # We skip starting weeks that are at the patient's known maximum
            if curWeek == maxes.loc[patient]:
                continue
            else:
                small_urn = np.array(data.loc[(data.Patient == patient) &
                                              (data.Weeks > curWeek)].index)
                target = data.iloc[random.choice(small_urn)]
                draw.at[self.y] = target.loc[self.y]
                output = output.append({**draw, 'targetWeek':target.Weeks
                                       }, ignore_index=True)
        return output.reset_index(drop=True)
    
    
    def split_from_target(self, data):
        return data.loc[:,~data.columns.isin(['Patient', self.y])], data.loc[:, self.y]
    
    
    def preprocess(self, data, cat_vars=['Sex','SmokingStatus'], scale_vars=['Percent','Age']):
        if self.firstrun:
            scale_cols = pd.DataFrame(self.scaler.fit_transform(data[scale_vars]), index=data.index)
            cat_cols = pd.DataFrame(self.cat_encoder.fit_transform(data[cat_vars]), index=data.index)
            self.firstrun = False
        else:
            scale_cols = pd.DataFrame(self.scaler.transform(data[scale_vars]), index=data.index)
            cat_cols = pd.DataFrame(self.cat_encoder.transform(data[cat_vars]), index=data.index) 
        scale_cols.columns = scale_vars
        cat_cols.columns = [s.replace(" ","").replace("-","") 
                            for s in np.concatenate(self.cat_encoder.categories_)]
        data = data.drop(np.concatenate([cat_vars, scale_vars]), axis=1)
        data = pd.concat([data, scale_cols, cat_cols], axis=1)
        return data.loc[:,np.sort(data.columns)]
        
        
    def fit(self, train, valid, sample_size, plot=False, early_stop=5, verbose=False, **kwargs):
        train, valid = self.sampler(train, sample_size), self.sampler(valid, sample_size)
        train, valid = self.preprocess(train), self.preprocess(valid)
        X_train, y_train = self.split_from_target(train)
        X_valid, y_valid = self.split_from_target(valid)
        self.model.fit(X_train, y_train,
                       eval_set=[(X_valid, y_valid)],
                       eval_metric='rmsle', 
                       early_stopping_rounds=early_stop,
                       verbose=verbose)
        print(self.model.best_score)
        if plot:
            plt.plot(self.model.evals_result()['validation_0']['rmsle'])
            plt.show()            
            
    def predict(self, newdata):
        newdata = self.preprocess(newdata)
        newdata = newdata.drop("Patient", axis=1)
        return self.model.predict(newdata)

n, r, k = 800, 0.18, 10
model = FibrosisModel(XGBRegressor,
                      n_estimators = n, 
                      learning_rate = r,
                      kernels = k)
model.fit(train, valid, 5000, plot=True)


/tmp/ipykernel_11/3954115808.py:17: FutureWarning: The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  maxes = data.groupby('Patient').Weeks.agg(max)
/tmp/ipykernel_11/3954115808.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  draw.at[self.y] = target.loc[self.y]


AttributeError: 'DataFrame' object has no attribute 'append'

In [3]:
class CustomLM():
    
    
    def __init__(self, X, y, a):
        self.model = Ridge(alpha=a, solver="cholesky")
        if X.shape[1] > 0:
            self.model.fit(X, y)
        else:
            self.pred = np.mean(y)
    
    
    def predict(self, X):
        if X.shape[1] > 0:
            return self.model.predict(X)
        else:
            return np.repeat(self.pred, X.shape[0])
        

        
class ResidualSimParameters():
    
    
    def __init__(self, model, data, max_delta=100, alpha=.1,
                 min_week=MIN_TEST_WEEK, max_week=MAX_TEST_WEEK):
        self.model = model
        self.data = data
        self.alpha = alpha
        self.linear_models = {}
        self.variances = {}
        self.patients = np.array(data.Patient.unique())
        # Compute the week spreads of each patient
        spreads = np.array([self.get_spread(data, patient) for patient in self.patients])
        df = pd.DataFrame({
            'patient': self.patients[np.argsort(spreads)[::-1]],
            'spread': np.sort(spreads)[::-1]
        })
        # Compute the parameters
        self.get_residual_params(df, min_week, max_week, max_delta)     
        
        
    def get_spread(self, data, patient):
        subset = data.loc[data.Patient == patient]
        return subset.Weeks.max() - subset.Weeks.min()
        
        
    def get_residual_params(self, df, min_week, max_week, max_delta):
        print("Getting residuals for delta ==", end=" ")
        X = []
        delta = min(df.spread.max(), max_delta)
        for d in np.arange(delta, 0, -1):
            i = max(np.where(df.spread >= d)[0]) 
            subset = df.iloc[:(i+1)]
            if np.maximum(0, subset.spread - d + 1).sum() > d:
                print(d, end="  ")
                X = self.get_residual_param_set(subset, d, X)
            d -= 1
    
    
    def get_residual_param_set(self, subset, delta, X):
        start_i = 0 if len(X) == 0 else X.shape[0]
        df = self.data
        
        for patient in np.array(subset.patient)[start_i:]:
            n = int(subset.loc[subset.patient == patient].spread) - delta + 1
            start_week = self.data.loc[self.data.Patient == patient].Weeks.min()
            
            # For each patient, obtain relevant residuals
            for i in range(n):                    
                # Define applicable weeks
                targetWeeks = np.arange(start_week + i, start_week + i + delta) + 1                
                # Create prediction variables DataFrame to feed model
                first_index = self.data.loc[(self.data.Patient == patient) &
                                            (self.data.Weeks == start_week + i)].index[0]                
                pred_df = self.data.iloc[np.repeat(first_index, delta)
                                        ].reset_index(drop=True).assign(targetWeek = targetWeeks)                
                # Make predictions and compare to labels
                preds = self.model.predict(pred_df.drop(TARGET_VAR, axis=1))                
                labels = self.data.loc[(self.data.Patient == patient) &
                                       (self.data.Weeks.isin(targetWeeks))
                                      ].sort_values(by='Weeks', ascending=False).deltaFVC
                error_vec = np.array(preds - labels)
                # Append to errors matrix
                if len(X) == 0:
                    X = np.array([error_vec])
                else:
                    prev_len = X.shape[0]
                    X = np.append(X, error_vec).reshape(prev_len + 1, delta)
                    
        # Fit a linear model to observed prediction errors
        X_, y_ = X[:,:-1], X[:,-1]
        lm = CustomLM(X_, y_, self.alpha)
        self.linear_models[delta] = lm
        residuals = lm.predict(X_) - y_
        self.variances[delta] = sum(residuals ** 2) / (len(y_) - 2)
        return X_ # The matrix can be recycled on the next delta; X's right-most column becomes its y
            
            
sim_parameters = ResidualSimParameters(model, train, alpha=1, max_delta=20)
print("Done.")


Getting residuals for delta == 20  

/tmp/ipykernel_11/3722038924.py:64: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  n = int(subset.loc[subset.patient == patient].spread) - delta + 1
/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


NotFittedError: need to call fit or load_model beforehand

In [4]:
def simulate_relative_errors(start_week, sim_params, n=1000,
                             max_week=MAX_TEST_WEEK):
    
    lm, variance = sim_params.linear_models, sim_params.variances
    max_delta = max(lm.keys())
    steps = np.arange(max_week - start_week) + 1
    error_matrix = np.array([])
    
    for s in steps:
        res_param_key = min(s, max_delta)
        if s == 1:
            base = np.zeros(n).reshape(-1,1)[:,1:]
        else:
            base = error_matrix[:, max(0, s - max_delta):]
        mu = lm[res_param_key].predict(base)
        sd = np.repeat(variance[res_param_key] ** 0.5, n)
        errors = np.random.normal(size=(n,1), loc=mu, scale=sd)
        if s == 1:
            error_matrix = errors
        else:
            error_matrix = np.concatenate([error_matrix, errors], axis=1)
            
    cum_error_matrix = np.cumprod(1 + error_matrix, axis =1) - 1  
    avg_cum_error = np.mean(cum_error_matrix, axis=0)
    
    return avg_cum_error


In [5]:
test = pd.read_csv(DATA_PATH + "test.csv")
test.head()


,Patient,Weeks,FVC,Percent,Age,Sex,SmokingStatus
0,ID00014637202177757139317,0,3807,90.076661,56,Male,Ex-smoker
1,ID00019637202178323708467,13,2100,92.858722,83,Female,Ex-smoker
2,ID00047637202184938901501,2,3313,89.929425,68,Male,Ex-smoker
3,ID00082637202201836229724,19,2918,99.794802,49,Female,Currently smokes
4,ID00126637202218610655908,18,2375,59.375000,78,Male,Ex-smoker


In [6]:
def predict_all(model, data, lb=MIN_TEST_WEEK, ub=MAX_TEST_WEEK, default_conf=200):
    
    weeks = np.arange(lb, ub+1)
    patients = data.Patient.unique()
    output = data.iloc[np.repeat(data.index, len(weeks))].reset_index(drop=True)
    output = output.assign(targetWeek = np.concatenate([weeks for _ in patients]))
    output = output.assign(predFVC = 0,
                           Confidence = default_conf)
    for patient in patients:
        start_week = data.loc[data.Patient == patient].Weeks.iloc[0]
        start_FVC = data.loc[data.Patient == patient].FVC.iloc[0]
        output.at[(output.Patient == patient) & 
                  (output.targetWeek <= start_week), 'predFVC'] = start_FVC
        pred_subset = output.loc[(output.Patient == patient) & 
                                 (output.targetWeek > start_week),
                                 ~output.columns.isin(['predFVC', 'Confidence'])]
        preds = start_FVC * np.cumprod(1 + model.predict(pred_subset))
        output.at[(output.Patient == patient) & 
                  (output.targetWeek > start_week), 'predFVC'] = preds        
        
    output = output.drop([c for c in data.columns if c not in ['Weeks', 'Patient']], axis=1)
        
    return output

output = predict_all(model, test)
output.head()


InvalidIndexError: You can only assign a scalar value not a <class 'numpy.int64'>

In [7]:
def adjust_confidence(pred_df, sim_parameters):
    
    for patient in pred_df.Patient.unique():
        start_week = pred_df.loc[pred_df.Patient == patient].Weeks.min()
        expected_rel_errors = simulate_relative_errors(start_week, sim_parameters)
        
        target_subset = pred_df.loc[(pred_df.Patient == patient) &
                                    (pred_df.targetWeek > start_week)].index
        confidence = expected_rel_errors * pred_df.predFVC.iloc[target_subset]
        confidence = np.minimum(1000, np.absolute(confidence)) * 2 ** 0.5
        pred_df.at[target_subset, 'Confidence'] = confidence
        
    return pred_df

output_w_conf = adjust_confidence(output, sim_parameters)


NameError: name 'output' is not defined

In [8]:
def finalize_format(df):
    df = df.assign(Patient_Week = df.Patient + '_' + df.targetWeek.astype(str))
    df = df.rename(columns={'predFVC':'FVC'})
    return df.drop(['Patient', 'targetWeek', 'Weeks'], axis=1)

final = finalize_format(output_w_conf)
final.to_csv("submission.csv", index=False)
final.head()


NameError: name 'output_w_conf' is not defined